# Домашнее задание 2 (опциональное).

В этом домашнем задании нужно будет реализовать метрики, бейзлайн со случайными рекомендациям, подобрать параметры для модели ALS из библиотеки implicit и сделать выводы по результатам экспериментов.  

Дедлайн выполнения 23 мая в 23:59.

[Форма](https://docs.google.com/forms/d/e/1FAIpQLSe0hc4Ta6AShEJYkxqzobvM2EZBbTknkzncpvNg_51HPJdXmQ/viewform?usp=dialog)  для отправки решения.

In [1]:
%%time
!pip install -q implicit

CPU times: user 1.77 s, sys: 253 ms, total: 2.02 s
Wall time: 10.7 s


In [62]:
import pandas as pd
import numpy as np

K = 10
SEED = 17
np.random.seed(SEED)

В этом задании будем использовать данные ML-1M.

In [2]:
data = pd.read_csv(
    "https://raw.githubusercontent.com/sb-ai-lab/RePlay/refs/heads/main/examples/data/ml1m_ratings.dat",
    sep="\t",
    names=["user_id", "item_id", "rating", "timestamp"]
)
assert data.shape == (1000209, 4)
data.head(2)

,user_id,item_id,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109


In [52]:
def get_log_info(log, user_column='user_id', item_column='item_id'):
    print(f"количество пользователей: {data[user_column].nunique()}")
    print(f"количество айтемов: {data[item_column].nunique()}")
    print(f"количество взаимодействий: {data.shape[0]}")

In [53]:
get_log_info(data)

количество пользователей: 6040
количество айтемов: 3706
количество взаимодействий: 1000209


# Задание 1: метрики и разбиение данных (1 балл)

### Разбиение данных (0.4 балла)

Разбейте данные по времени.
- Первые 80% данных по timestamp отложите в обучающую выборку, следующие 10% - в валидационную, последние 10% в тестовую выборку.
- Удалите холодных (отсутствующих в обучающей выборке) пользователей из валидационной и тестовой выборки.
- В валидационной и тестовой выборке оставьте только положительные оценки, положительными будем считать $r_{ij}>3$.

In [ ]:
# TODO: your code

### метрики (0.6 балла)

Реализуйте метрику mrr@10 (0.3 балла).

MRR@k для пользователя считается как обратный ранг первого релевантного айтема в рекомендациях
$$MRR@k(u) = \frac{1}{rank_j}$$

In [ ]:
def mrr_user(row, k):
    """row: row of pd.Dataframe with columns `pred_list`, `gt_list`"""
    # TODO: your code
    return


assert np.allclose(mrr_user(pd.Series({'user_id': 1, 'pred_list': [1, 2], 'gt_list': [4, 5, 2, 6]}), k=2), 0.5, atol=1e-3)
assert np.allclose(mrr_user(pd.Series({'user_id': 1, 'pred_list': [1, 2], 'gt_list': [4, 5, 2, 6]}), k=1), 0., atol=1e-3)
assert np.allclose(mrr_user(pd.Series({'user_id': 1, 'pred_list': [], 'gt_list': [4, 5, 2, 6]}), k=10), 0)
assert np.allclose(mrr_user(pd.Series({'user_id': 1, 'pred_list': [1, 3], 'gt_list': []}), k=10), 0)

Общая метрика MRR@k получается усреднением по пользователям
$$MRR@k = \frac{\sum\limits_{u=1}^{n_{users}} MRR@k(u)}{n_{users}}$$

Реализуйте метрику ndcg@10 (0.3 балла).

$$DCG@k(u) = \sum\limits_{j=1}^{k}\frac{2^{r_{uj}} - 1}{\log_2{(j+1})},\ nDCG@k(u) = \frac{DCG@k(u)}{IDCG@k(u)},\ IDCG@k(u) = \max (DCG@k(u))$$

In [ ]:
def ndcg_user(row, k):
    """row: row of pd.Dataframe with columns `pred_list`, `gt_list`"""
    # TODO: your code
    return


assert np.allclose(ndcg_user(pd.Series({'user_id': 1, 'pred_list': [1, 2], 'gt_list': [3, 4, 2, 6]}), k=2), 0.387, atol=1e-3)
assert np.allclose(ndcg_user(pd.Series({'user_id': 1, 'pred_list': [1, 2], 'gt_list': [3, 4, 2, 6]}), k=1), 0., atol=1e-3)
assert np.allclose(ndcg_user(pd.Series({'user_id': 1, 'pred_list': [], 'gt_list': [4, 5, 2, 6]}), k=10), 0)
assert np.allclose(ndcg_user(pd.Series({'user_id': 1, 'pred_list': [1, 3], 'gt_list': []}), k=10), 0)


$$nDCG@k = \frac{\sum\limits_{u=1}^{n_{users}} nDCG@k(u)}{n_{users}}$$

# Задание 2: ALS (1 балл)

### Бейзлайн (0.2 балла)

Реализуйте бейзлайн делающий случайные предсказания для каждого пользователя.
- на выходе используйте pd.DataFrame с колонками user_id, item_id, relevance
- сделайте предсказание на тестовой выборке и замерьте по нему значения метрик mrr@10 и ndcg@10

In [ ]:
# TODO: your code

In [59]:
assert random_baseline.shape[0] == test['user_id'].nunique() * K
random_baseline.head(2)

,user_id,item_id,relevance
0,1,323,1.0
1,1,1159,1.0


### ALS (0.8 балла)

- (0.2 балла) обучите ALS из библиотеки [implicit](https://github.com/benfred/implicit) на обучающей выборке, используя все оценки. Используйте последнюю версию библиотеки. Получите предсказания на тестовой выборке, используйте при этом фильтрацию просмотренных айтемов. Замерьте метрики mrr@10 и ndcg@10 по полученным предсказаниями.
- (0.4 балла) подберите на валидационной выборке параметры `factors`, `regularization`, `alpha` например перебором по сетке или при помощи [optuna](https://github.com/optuna/optuna)
- (0.1 балла) постройте график зависимости ndcg@10 от количества факторов
- (0.1 балла) сделайте выводы по результатам экспериментов.

Полезные ссылки:
- http://www.benfrederickson.com/matrix-factorization/
- [документация библиотеки implicit](https://benfred.github.io/implicit/api/models/index.html)
- [примеры работы библиотеки implicit](https://github.com/benfred/implicit/blob/main/examples/tutorial_lastfm.ipynb). Обратите внимание что implicit.datasets.lastfm.get_lastfm возвращает CSR sparse матрицу где по строкам айтемы, а по столбцам юзеры.

In [ ]:
# TODO: your code